In [137]:
import sympy as sym
import numpy as np
from scipy import constants
from IPython.display import Math, display

constants.pi

3.141592653589793

An sich möchte ich hier nur etwas experimentieren, wie es aussehen könnte, wenn ich später ein einheitliches File habe, aus dem ich alle meine Variablen usw. nehme. Ein Beispiel davon wie das ausshene könnte ist hier

In [138]:
x = {
    "name": "nominal state",
    "description": "",
    "definition": "",
    "components": [
        {
            "name": "IMU position",
            "code variable": "p",
            "latex symbol": "p",
            "dimensionality": 3,
        },
        {
            "name": "IMU orientation",
            "code variable": "q",
            "latex symbol": "q",
            "dimensionality": 4,
        },
        # ... in between stuff ...
        {
            "name": "IMU gyro bias",
            "code variable": "omega_b",
            "latex symbol": r"\omega_{b}",
            "dimensionality": 3,
        },
        {
            "name": "Speaker position",
            "code variable": "p_s",
            "latex symbol": "p_{s}",
            "dimensionality": 3,
        },
    ],
}

# Stellen, die bisher doof laufen

## jupyter notebook

Ich glaube ganz grundsätzlich kann ich einen Teil der Funktionen, die ich da als preliminaries habe, einfach nehmen und in ein Allgemeines Dokument packen. Darunter z.B.

In [139]:
def R(q):
    qw, qx, qy, qz = q[0], q[1], q[2], q[3]
    return sym.Matrix(
        [
            [1 - 2 * (qy**2 + qz**2), 2 * (qx * qy - qz * qw), 2 * (qx * qz + qy * qw)],
            [2 * (qx * qy + qz * qw), 1 - 2 * (qx**2 + qz**2), 2 * (qy * qz - qx * qw)],
            [2 * (qx * qz - qy * qw), 2 * (qy * qz + qx * qw), 1 - 2 * (qx**2 + qy**2)],
        ]
    )

Gerade die Konstanten wie gravity, gravity_vec und die IMU->cam translations können auch hier rein, die substitution für Observability kann ich ja später immer noch im Notebook machen. Hierbei könnte es sinnvoll sein, die values einmal für sympy und einmal für die konkrete Anwendung zu definieren. Weil in sympy könnte es praktisch sein, bei manchen Umformungen noch die trigonometrischen Eigenschaften der Funktion zu erhalten, während im Code es später eher nicht mehr so eine große Relevanz hat

In [140]:
gravity_constant = 9.80665
gravity_vector = [0, 0, -1 * gravity_constant]

pi = 3.141592653589793


# Das für sympy/notebook
Phi = 102 * (constants.pi / 180)
# subs_IMU_cam_rotation = {
# IMU_cam_w_rotation: sym.cos(-Phi / 2),
# IMU_cam_x_rotation: sym.sin(-Phi / 2),
# }

# Und das alles dann nochmal für numpy

Dann kommt der eigentliche Variablenteil. Zum einen möchte ich gerne die latex Variablen alle irgendwo haben, sodass ich sie nur noch für meine Sympy Symbole importieren muss. Hier ein Beispiel, genommen aus dem anfänglichen dictionary Snippet:

In [ ]:
# alte Implementierung
omega_b_old = sym.Matrix(
    sym.symbols(r"\omega_{b\,x} \omega_{b\,y} \omega_{b\,z}", real=True)
)  # gyroscope bias


# neue Implementierung
def get_component(target_key, target_value, node):
    """Recursively search the schema tree for an element matching target_key == target_value."""
    if isinstance(node, dict):
        if node.get(target_key) == target_value:
            return node
        for val in node.values():
            result = get_component(target_key, target_value, val)
            if result:
                return result
    elif isinstance(node, list):
        for item in node:
            result = get_component(target_key, target_value, item)
            if result:
                return result
    return None


# Ah ne, eigentlich haben die ja später alle garkeine subscript drin, sondern z.b. p ist einfach nur so, ganz ohne subscript
# Ich muss einfach sicherstellen das subscripts immer am Ende stehen (nach superscripts) und in geschweiften Klammern stehen
# im allerschlimmsten Fall macht das auch nur den Latex Print kaputt, wie das da später geprinted wird, im code sollte das keinen Schaden anrichten können, wenn der string nicht perfekt ist
# das ist noch nicht ganz richtig, ich muss ja kein Bracket zumachen wenn ich später
def add_dimensions_in_markdown(var_name, dim):

    # Ich muss einerseits checken, ob es schon ein subscript gibt, in dem Fall adde ich die dimension innerhalb der Klammern
    # Wenn es noch kein subscript gibt, dann erstelle ich eins mit der dimension in curly brackets
    if var_name[-1] == "}":
        var_name = var_name[:-1]
    if dim == 3:
        return [var_name + r"\," + f"{i}" + "}" for i in ["x", "y", "z"]]
    elif dim == 4:
        return [var_name + r"\," + f"{i}" + "}" for i in ["w", "x", "y", "z"]]
    else:
        print("everything is fucked")
        return None


def expand_latex_symbols(var_name, dim):
    if dim not in (3, 4):
        return
    dimensions = ["x", "y", "z"] if dim == 3 else ["w", "x", "y", "z"]
    if "_" in var_name:
        return [var_name[:-1] + r"," + i + r"}" for i in dimensions]
    else:
        return [var_name + r"_{" + i + r"}" for i in dimensions]


test = get_component("name", "Speaker position", x)
array = expand_latex_symbols(test["latex symbol"], test["dimensionality"])
# array = add_dimensions_in_markdown(test["latex symbol"], test["dimensionality"])

# omega_b_new = sym.Matrix(sym.symbols(array, real=True))

# display(omega_b_old.T, omega_b_new.T)
array

['p_{s,x}', 'p_{s,y}', 'p_{s,z}']

In [142]:
testarray = [None] * 15
accel_bias = slice(4, 7)
testarray[4:7] = [12, 13, 14]
testarray, testarray[accel_bias]

([None,
  None,
  None,
  None,
  12,
  13,
  14,
  None,
  None,
  None,
  None,
  None,
  None,
  None,
  None],
 [12, 13, 14])

Kann ich auch die Länge eines Arrays in einem dictionary erstellen, abhängig von Daten innerhalb des dictionaries?

In [143]:
def fill_out_components(full_vector):
    current_index = 0
    for component in full_vector["components"]:
        dim = component["dimensionality"]
        component["array_position"] = slice(current_index, current_index + dim)
        current_index += dim


# Run the setup
fill_out_components(x)
x

{'name': 'nominal state',
 'description': '',
 'definition': '',
 'components': [{'name': 'IMU position',
   'code variable': 'p',
   'latex symbol': 'p',
   'dimensionality': 3,
   'array_position': slice(0, 3, None)},
  {'name': 'IMU orientation',
   'code variable': 'q',
   'latex symbol': 'q',
   'dimensionality': 4,
   'array_position': slice(3, 7, None)},
  {'name': 'IMU gyro bias',
   'code variable': 'omega_b',
   'latex symbol': '\\omega_{b}',
   'dimensionality': 3,
   'array_position': slice(7, 10, None)},
  {'name': 'Speaker position',
   'code variable': 'p_s',
   'latex symbol': 'p_{s}',
   'dimensionality': 3,
   'array_position': slice(10, 13, None)}]}

In [144]:
# Nur für später, falls ich doch einmal nicht mit den slices arbeiten möchte
get_component("code variable", "q", x)["array_position"].start
get_component("code variable", "q", x)["array_position"].stop

7

Ich sollte überlegen, ob ich später meine Variablen ein bisschen besser benenne, im Sinne von kürzer. Das würde viele Ausdrücke sehr viel leserlicher machen.

Sowieso ist langfristig die Frage, ob ich wirklich alles im notebook machen möchte und dann später pickeln. Aber da denke ich durchaus, dass das in Teilen sinnvoll sein kann.

## python scripts

Also an sich habe ich das Gefühl, ich habe fast alle nützlichen Sachen für einen stabilen Aufbau für meine Notebook Erstellung zusammengesammelt. Jetzt ist noch die Frage ob ich zusätzliche Dinge für meine python scripte brauche.

alternative Titel für meine keys in den dictionaries

Current Key,Alternative Options (Short & Descriptive),Best Pick
"""name""","""title"", ""id"", ""label""","""title"" or ""id"""
"""description""","""desc"", ""info"", ""notes""","""desc"""
"""definition""","""defn"", ""math"", ""formula""","""defn"""
"""components""","""comps"", ""parts"", ""elements""","""comps"""

Current Key,Alternative Options (Short & Descriptive),Best Pick
"""name""","""id"", ""label"", ""name""","""name"" (or ""id"" if unique)"
"""code variable""","""var"", ""sym"", ""code""","""var"""
"""markdown symbol""","""tex"", ""math"", ""symbol""","""tex"" (or ""math"")"
"""dimensionality""","""dim"", ""size"", ""len""","""dim"""


und ich könnte auch eine Funktion schreiben, welche dieses wiederholende pattern auch wirklich repetitiv Einsetzt, und nicht so das ich von hand ausversehen Fehler mache

In [188]:
# dictionary Erstellung
def create_comp(name, var, tex, dims):
    if (
        isinstance(name, str)
        and isinstance(var, str)
        and isinstance(tex, str)
        and isinstance(dims, int)
    ):
        return {"name": name, "var": var, "latex": tex, "dims": dims}


x = {
    "name": "nominal state",
    "info": "",
    "components": [
        create_comp(*c)
        for c in [
            ["IMU position", "p", r"p", 3],
            ["IMU orientation", "q", r"q", 4],
            ["IMU gyro bias", "omega_b", r"\omega_{b}", 3],
            ["Speaker position", "p_s", r"p_{s}", 3],
        ]
    ],
}


def fill_out_components(full_vector):
    current_index = 0
    for component in full_vector["components"]:
        dim = component["dims"]
        component["slice"] = slice(current_index, current_index + dim)
        current_index += dim


# Run the setup
display(x)
fill_out_components(x)
display(x)

get_component("var", "q", x)

{'name': 'nominal state',
 'info': '',
 'components': [{'name': 'IMU position', 'var': 'p', 'latex': 'p', 'dims': 3},
  {'name': 'IMU orientation', 'var': 'q', 'latex': 'q', 'dims': 4},
  {'name': 'IMU gyro bias',
   'var': 'omega_b',
   'latex': '\\omega_{b}',
   'dims': 3},
  {'name': 'Speaker position', 'var': 'p_s', 'latex': 'p_{s}', 'dims': 3}]}

{'name': 'nominal state',
 'info': '',
 'components': [{'name': 'IMU position',
   'var': 'p',
   'latex': 'p',
   'dims': 3,
   'slice': slice(0, 3, None)},
  {'name': 'IMU orientation',
   'var': 'q',
   'latex': 'q',
   'dims': 4,
   'slice': slice(3, 7, None)},
  {'name': 'IMU gyro bias',
   'var': 'omega_b',
   'latex': '\\omega_{b}',
   'dims': 3,
   'slice': slice(7, 10, None)},
  {'name': 'Speaker position',
   'var': 'p_s',
   'latex': 'p_{s}',
   'dims': 3,
   'slice': slice(10, 13, None)}]}

{'name': 'IMU orientation',
 'var': 'q',
 'latex': 'q',
 'dims': 4,
 'slice': slice(3, 7, None)}